# 第 58 课｜Docker 可复现研究环境设计

本 Notebook 对应「365 天量化金融」第 58 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/58_youtube-dnU4YmbMe68/index.html`
- 环境：`pkm-quant`
- 数据：生成 Dockerfile 与忽略清单，不启动 Docker
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

Docker 镜像封装 Linux 用户空间、Python 和依赖；数据、密钥和输出通过挂载或 secret 进入。Notebook 生成并静态检查最小 Docker 配置，不假设本机已经启动 Docker Desktop。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 1. Dockerfile 的分层顺序影响缓存

先复制依赖锁文件并安装，后复制代码。只改代码时可以复用依赖层。镜像内使用非 root 用户，运行命令只生成本地报告。

In [2]:
# 生成最小研究镜像配置。
dockerfile = '''FROM python:3.11-slim

WORKDIR /app
COPY requirements.lock /app/requirements.lock
RUN pip install --no-cache-dir -r requirements.lock

RUN useradd --create-home quant
COPY --chown=quant:quant src/ /app/src/
USER quant

CMD ["python", "-m", "src.monthly_report"]
'''
dockerfile_path = OUTPUT_DIR / "58_Dockerfile.example"
dockerfile_path.write_text(dockerfile, encoding="utf-8")
print(dockerfile)
assert dockerfile.index("COPY requirements.lock") < dockerfile.index("COPY --chown=quant:quant src/")
assert "USER quant" in dockerfile

FROM python:3.11-slim

WORKDIR /app
COPY requirements.lock /app/requirements.lock
RUN pip install --no-cache-dir -r requirements.lock

RUN useradd --create-home quant
COPY --chown=quant:quant src/ /app/src/
USER quant

CMD ["python", "-m", "src.monthly_report"]



## 2. 原始数据、输出和密钥不进入镜像

`.dockerignore` 缩小构建上下文；真实数据通过只读挂载 `/data:ro`，输出挂载 `/outputs`。密钥不写入 Dockerfile 或镜像层。

In [3]:
# 生成 .dockerignore 与可复制的 build/run 命令。
dockerignore = '''.git
.env
.venv
__pycache__
data/
outputs/
*.mp4
*.sqlite
'''
ignore_path = OUTPUT_DIR / "58_dockerignore.example"
ignore_path.write_text(dockerignore, encoding="utf-8")
commands = pd.Series({
    "build": "docker build -f 58_Dockerfile.example -t pkm-quant:p2 .",
    "run": "docker run --rm -v $PWD/data:/data:ro -v $PWD/outputs:/outputs pkm-quant:p2",
}, name="command")
display(commands.to_frame())
assert ".env" in dockerignore
assert "/data:ro" in commands["run"]

,command
build,docker build -f 58_Dockerfile.example -t pkm-q...
run,docker run --rm -v $PWD/data:/data:ro -v $PWD/...


## 3. 静态安全检查

检查 Dockerfile 没有复制 `.env`、原始数据或 outputs，也没有使用 `ARG SECRET`。这是最低限度检查，不替代镜像扫描。

In [4]:
# 对生成配置执行确定性的静态规则。
forbidden = ["COPY .env", "COPY data", "COPY outputs", "ARG SECRET", "ENV API_KEY"]
violations = [token for token in forbidden if token in dockerfile]
audit = pd.Series({
    "forbidden_instructions": len(violations),
    "non_root_user": "USER quant" in dockerfile,
    "dependency_lock_first": dockerfile.index("requirements.lock") < dockerfile.index("src/"),
    "data_ignored": "data/" in dockerignore,
    "secrets_ignored": ".env" in dockerignore,
})
display(audit.to_frame("result"))
assert not violations
assert audit.drop("forbidden_instructions").all()

,result
forbidden_instructions,0
non_root_user,True
dependency_lock_first,True
data_ignored,True
secrets_ignored,True


## 方法边界与复用结论


- 容器共享宿主机内核，不等于虚拟机；macOS 上还经过 Linux 虚拟化层。
- 相同镜像不能消除 CPU 架构、线程库、时区和外部数据变化。
- 生产镜像还应固定摘要 digest、扫描漏洞，并以 secret 管理敏感配置。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。